# データの読み込みおよび確認

In [ ]:
import pandas as pd
df = pd.read_csv('bike.tsv', sep='\t')
df.head(5)

In [ ]:
labele_means = {'dteday':'日付','weekday':'曜日(0=日,1=月,2=火,3=水,4=木,5=金,6=土)',
    'weather_id':'天気','holiday':'祝日フラグ(普通の土日は含めない)',
    'workingday':'平日フラグ','cnt':'利用者数'}
for label, mean in labele_means.items():
    print(f'{label}={mean}')

In [ ]:
weather = pd.read_csv('weather.csv', encoding = 'shift-jis')
weather.head(5)

In [ ]:
df2 = df.merge(weather, how='inner', on='weather_id')
# print(df2.head(5))
df2 = df2.sort_values(by='dteday')
print(df2.head(5))

In [ ]:
print('\nmean:',df2.groupby('weather')['cnt'].mean(),'\n')
print('\nmin:',df2.groupby('weather')['cnt'].min(),'\n')
print('\nmax:',df2.groupby('weather')['cnt'].max(),'\n')

In [ ]:
temp = pd.read_json('temp.json')
temp.head(2)

In [ ]:
temp = temp.T
temp

In [ ]:
temp.loc[199:201]

In [ ]:
df2[df2['dteday'] == '2011-07-20']

In [ ]:
df3 = df2.merge(temp, how = 'left', on = 'dteday')
df3.head(5)

In [ ]:
df3[df3['dteday'] == '2011-07-20']

# 気温と湿度の折れ線グラフ

In [ ]:
df3[['temp', 'hum']].plot(kind='line')

# 気温と湿度の度数分布図

In [ ]:
df3['temp'].plot(kind='hist')
df3['hum'].plot(kind = 'hist', alpha = 0.5)

# 平均気温？(atemp)の欠損値を確認する

In [ ]:
df3['atemp'].plot(kind='line')
df3['atemp'].loc[695:705]

In [ ]:
df3['atemp'].loc[695:705].plot(kind='line')

# 平均気温？(atemp)の欠損値を線形補間する

In [ ]:
df3['atemp'] = df3['atemp'].astype(float) # atempはobject型なのでfloat型に変更
df3['atemp'] = df3['atemp'].interpolate()
df3.loc[695:705,'atemp'].plot()

In [ ]:
from sklearn.covariance import MinCovDet

df4 = df3.loc[:, 'atemp':'windspeed']
df4 = df4.dropna() # 欠損値を削除
# マハラノビス距離を計算するための準備
mcd = MinCovDet(random_state = 0, support_fraction=0.7)
mcd.fit(df4) # マハラノビス距離を計算するために必要な、共分散行列というものを計算

# マハラノビス距離を計算
distance = mcd.mahalanobis(df4)
distance

In [ ]:
distance = pd.Series(distance) # numpy配列をpandasのシリーズに変換
distance.plot(kind='box')

In [ ]:
tmp= distance.describe() # さまざまな基本統計量を計算
tmp

# 四分位範囲を用いた外れ値の判定

In [ ]:
iqr = tmp['75%'] - tmp['25%'] # IQR計算
jougen = 1.5 * iqr + tmp['75%'] # 上限値
kagen = tmp['25%'] - 1.5*iqr # 下限値

# 上限と下限の条件をもとにシリーズで条件検索
outliner = distance[(distance > jougen) | (distance < kagen)]
outliner

In [ ]:
import matplotlib.pyplot as plt
df4.plot(kind='scatter', x='atemp', y='hum')
plt.savefig('test0.png')

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2,2,figsize=(10,6)) # 1枚の画像を2行2列に分割、サイズは縦6横10

df4.plot(kind='scatter', x='atemp',y='hum',ax=axs[0,0]) # 画像内の0行0列の位置に配置
df4.plot(kind='scatter', x='temp',y='hum',ax=axs[1,1]) # 画像内の0行0列の位置に配置
